#  1.SummarizationMiddleware中间件
## 举例1：测试trigger、keep参数

In [ ]:
import os

from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.messages import HumanMessage
from langchain_deepseek import ChatDeepSeek
from dotenv import load_dotenv
from scripts.regsetup import description
from rich import print as rprint
#1.读取.env配置文件信息,相关的环境变量以.env文件中的优先
load_dotenv(verbose=True)
DEEPSEEK_API_KEY=os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL=os.getenv("DEEPSEEK_BASE_URL")
#2.模型初始化
model=ChatDeepSeek(
    model="deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    api_base=DEEPSEEK_BASE_URL,
    profile={
        "max_input_tokens": 1_000_000
    },
    # 关键修改：关闭思考模式
    extra_body={
        "thinking": {
            "type": "disabled"
        }
    },
)

In [ ]:
from langchain_core.messages import SystemMessage,AIMessage,HumanMessage
from langchain.agents.middleware import SummarizationMiddleware
messages = [
        SystemMessage("你是个非常友好的AI助手"),
        HumanMessage("你好啊，我是老王，你是谁？"),
        AIMessage("你好老王，我是小王"),
        HumanMessage("好的小王，很高兴认识你"),
        AIMessage("你高兴得太早了"),
        HumanMessage("呵呵，你什么意思")
]
agent=create_agent(
    model="deepseek-v4-flash",
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=[
                ("tokens",100),
                ("messages",6),
                ("fraction",0.001)
            ],
            keep=("messages",2)
        )
    ]
)
response =agent.invoke(
    {
        "messages":messages
    }
)
for msg in response["messages"]:
    msg.pretty_print()

## 举例2：summary_prompt

In [ ]:

from langchain_core.messages import SystemMessage, AIMessage, HumanMessage
from langchain.agents.middleware import SummarizationMiddleware

messages = [
    SystemMessage("你是个非常友好的AI助手"),
    HumanMessage("你好啊，我是老王，你是谁？"),
    AIMessage("你好老王，我是小王"),
    HumanMessage("好的小王，很高兴认识你"),
    AIMessage("你高兴得太早了"),
    HumanMessage("呵呵，你什么意思")
]
agent = create_agent(
    model="deepseek-v4-flash",
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=[
                ("tokens", 100),
                ("messages", 6),
                ("fraction", 0.001)
            ],
            keep=("messages", 2),
            summary_prompt="对历史消息摘要，消息列表如下\n{messages}"
        )
    ]
)
response = agent.invoke(
    {
        "messages": messages
    }
)
for msg in response["messages"]:
    msg.pretty_print()
